# Integrated Intelligent Systems: Tutorial 1 - Foundations (3 Pts)

This notebook introduces the fundamental components, challenges, and definitions, central to designing and implementing Cognition-Enabled Robot Agents.

## Part 1: The Unified Theoretical Exercise (E1–E6) (1.5 Pt)

**Title:** From Natural Language Request to action execution: The Eva Coffee Scenario

![Alternative Text](coffee-task-photo.png)

### Scenario Background
Eva requests: "Bring me my coffee." Her usual blue mug is in the dishwasher; a white mug is available on the counter. The fill level can vary, and Eva may shift position during execution.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)
Students work through this unified exercise in the VRB or on paper, structured across the six developmental steps of Lecture 1:

1. **E1 · Opening Judgment (Data vs. Competence):**
   - If an end-to-end vision-language-action model or a demonstration video shows a robot successfully serving coffee 100 times in a row, does that prove the robot "can do the task" across any home environment? State your reasoning and rate your confidence from 1 to 5.
     

2. **E2 · Changed Case Prediction (Physical Bounds):**
   - A half-full white mug was carried successfully along the route. Suppose the mug is now nearly full. Predict the physical consequence during a turn, and identify the untested assumption in the original run.
     

3. **E3 · The Commitment Decision (Warrant Gate):**
   - The blue mug is unavailable and white-mug acceptance is unconfirmed. Choose whether the robot should ask, observe, or proceed. State the single item of evidence that justifies your choice.
     

4. **E4 · The Five Questions of Understanding (T1–T5):**
   - Formulate one distinct question for each dimension—Explain ($\text{T1}$), Predict ($\text{T2}$), Design ($\text{T3}$), Bound ($\text{T4}$), and Check ($\text{T5}$)—concerning the coffee delivery episode. At least one question must address a changed condition that a video alone cannot answer.
     

5. **E5 · System Decomposition (Mapping to Architecture):**
   - List the core functional modules required to handle Eva's request (interpreting, perceiving, proposing, predicting, warranting, acting, and recording) and map them onto the Integration Diagram.
     

6. **E6 · Transfer & Exit Ticket (Justified Adaptation):**
   - Suppose Eva stands up while the robot is carrying the coffee. Revise the action, predict the potential failure mechanism, and state the exact evidence criterion required before the robot commits to the revised path.

**E1: Opening Judgment (Data vs. Competence)**
No, it does not prove that the robot can perform a task in any environment. One hundred successes on video only support a claim of success within those exact demonstrated conditions. To claim wider competence across different environments, a robot needs a declared range, varied tests, checked outcomes, and failure evidence.
*Confidence: 5*

**E2: Changed Case Prediction (Physical Bounds)**
* **Physical Consequences:** If the turn speed remains unchanged, carrying a nearly full mug will likely result in spilling coffee over the freeboard due to high lateral acceleration ($v^2/r$) during the turn.
* **Untested Assumption:** The original run relied on the untested assumption that the control method and motion plan (velocity) that succeeded for the half-full mug would automatically transfer to a higher fill level.

**E3: The Commitment Decision (Warrant Gate)**
* **Decision:** The robot should ask.
* **Evidence:** The single item of evidence required to proceed is an explicit verbal confirmation from Eva that the white mug is acceptable. Asking is necessary because, absent this confirmation, her acceptance is unknown, and merely observing the scene will not yield evidence of her internal preferences.

**E4: The Five Questions of Understanding (T1–T5)**
Based on the guiding questions for understanding a robot site:
* **T1 (Explain):** Why did the coffee not spill on the turn, was it due to a slow turn or because it was only a half-full mug?
* **T2 (Predict):** Will a nearly full mug spill on this specific route?
* **T3 (Design):** Would using a different gripper or adding a tray change the carry?
* **T4 (Bound):** For exactly which mugs, fill levels, and routes does this serving method work?
* **T5 (Check):** Did the episode record capture Eva's answer, the executed motion, and the final outcome?

**E5: System Decomposition (Mapping to Architecture)**
* **Interpret & Ground:** Determines which drink and which mug are needed.
* **Propose (System 1 - Learning):** Generates learned candidate actions for the task.
* **Perceive:** Converts raw sensor readings into a belief about the environment.
* **Know & Predict (System 2 - Warrant):** Uses a digital twin of the world to simulate consequences.
* **Decide (Warrant Gate):** Checks if the grounds hold here to authorize the proposed action.
* **Act & Regulate:** Executes the move, watches the execution, and repairs if needed.
* **Record & Justify:** Maintains a trace of what was asked, believed, committed, and observed so the run can be audited.

**E6: Transfer & Exit Ticket (Justified Adaptation)**
* **Revised Action:** The robot must pause its current movement and re-plan the handover approach.
* **Potential Failure Mechanism:** Handing over the coffee at the original planned height and trajectory (assuming she was seated) could cause a collision, drop, or spill because her posture and location have changed.
* **Exact evidence criterion:** Obtain a fresh pose estimate of Eva (e.g., sensor data age < 100ms, covariance within safe bounds), confirm the new handover pose is kinematically reachable, predict the slosh for the new trajectory is below the 1.5mm limit, and re-pass the Warrant Gate before moving.

## Part 2: Interactive Virtual Lab & Python Tutorial (1.5 Pt)

**Title:** Implementing the Embodied Cognitive Architecture

### Scenario Background
Eva requests: "Bring me my coffee."

---

In [1]:
#!/usr/bin/env python3
"""Embodied Cognitive Architecture
"""

from dataclasses import dataclass, field
from typing import Dict, Any, Tuple, Optional, List
import random
import time

@dataclass(frozen=True)
class GoalSpecification:
    """Formalized, concrete goal specification derived via semantic interpretation."""
    target_object: str
    destination: str
    target_volume_ml: float
    max_allowable_force_n: float
    max_slosh_wave_mm: float
    max_nav_velocity_mps: float
    time_frame_seconds: float

@dataclass
class ParameterPacket:
    """System 1 (The Proposer): Flexible sampler proposing execution parameters."""
    grasp_pose: Tuple[float, float, float]
    max_velocity_mps: float
    tilt_rate_radps: float
    source_generator: str = "stochastic_sampler"

class Proposer:
    """ To be completed """
    def __init__(self, base_velocity_mean: float = 0.5, base_tilt_mean: float = 0.1):
        self.base_velocity_mean = base_velocity_mean
        self.base_tilt_mean = base_tilt_mean

    def generate_proposal(self) -> ParameterPacket:
        # Gaussian noise reflects real-world imitation learning variances
        v_max = max(0.05, random.gauss(self.base_velocity_mean, 0.15))
        tilt = max(0.01, random.gauss(self.base_tilt_mean, 0.05))

        return ParameterPacket(
            grasp_pose=(0.45, 0.12, 0.30),
            max_velocity_mps=round(v_max, 3),
            tilt_rate_radps=round(tilt, 3)
        )


@dataclass
class Observation:
    """Raw, noisy sensor measurements at time t (distinct from internal belief)."""
    timestamp: float
    measured_temperature_c: float
    measured_friction: float
    eva_position: str
    room_busyness: str

@dataclass
class BeliefState:
    """The robot's current internal belief state (updated via perception)."""
    pouch_present: bool = True
    pouch_material: str = "flexible_plastic"
    fluid_viscosity_pas: float = 1.2
    vial_ready: bool = True
    camera_active: bool = True
    fallback_path_exists: bool = True
    audit_logger_enabled: bool = True
    # Real-time environmental parameters maintained internally
    ambient_temperature_c: float = 21.5
    surface_friction_coefficient: float = 0.45
    eva_position: str = "living_room"
    room_busyness: str = "medium"  # 'low', 'medium', 'high'
    preferred_cup: str = "blue_ceramic_mug"


class InterpretationSystem:
    """Uses a semantic knowledge base to map vague instructions and context
    into precise physical goal parameters (GoalSpecification).
    """
    def __init__(self, knowledge_base: Dict[str, Any]):
        self.kb = knowledge_base

    def interpret(self, raw_utterance: str, belief: BeliefState) -> GoalSpecification:
        print(f"\n[INTERPRETATION] Analyzing instruction: '{raw_utterance}' in context of Eva (Position: {belief.eva_position}, Room Busyness: {belief.room_busyness})")

        cup_type = belief.preferred_cup
        cup_properties = self.kb["cups"].get(cup_type, {"max_force": 5.0, "max_wave": 2.0})

        busyness_factor = self.kb["room_busyness_limits"].get(belief.room_busyness, 0.5)
        max_nav_vel = 1.2 * busyness_factor

        is_urgent = "schnell" in raw_utterance.lower() or "fast" in raw_utterance.lower()
        time_frame = 30.0 if is_urgent else 60.0
        target_volume = 15.0 if not is_urgent else 12.0

        goal = GoalSpecification(
            target_object=cup_type,
            destination=belief.eva_position,
            target_volume_ml=target_volume,
            max_allowable_force_n=cup_properties["max_force"],
            max_slosh_wave_mm=cup_properties["max_wave"],
            max_nav_velocity_mps=max_nav_vel,
            time_frame_seconds=time_frame
        )

        print(f" -> Derived Goal: Object={goal.target_object}, MaxForce={goal.max_allowable_force_n}N, MaxNavVel={goal.max_nav_velocity_mps}m/s, TimeFrame={goal.time_frame_seconds}s")
        return goal


class SensingSystem:
    """Simulates real-time noisy hardware sensors emitting raw observations."""
    def __init__(self):
        pass

    def sense(self, true_ambient_temp: float, true_friction: float, eva_pos: str, busyness: str) -> Observation:
        noisy_temp = true_ambient_temp + random.uniform(-0.5, 0.5)
        noisy_friction = true_friction + random.uniform(-0.05, 0.05)

        return Observation(
            timestamp=time.time(),
            measured_temperature_c=round(noisy_temp, 2),
            measured_friction=round(noisy_friction, 2),
            eva_position=eva_pos,
            room_busyness=busyness
        )


class PerceptionSystem:
    """Estimator module that updates the internal BeliefState using raw Observations."""
    def __init__(self, belief_state: BeliefState):
        self.belief_state = belief_state

    def update(self, observation: Observation) -> None:
        self.belief_state.ambient_temperature_c = observation.measured_temperature_c
        self.belief_state.surface_friction_coefficient = observation.measured_friction
        self.belief_state.eva_position = observation.eva_position
        self.belief_state.room_busyness = observation.room_busyness


class SemanticDigitalTwin:
    """System 2 (Semantic Digital Twin): Imperfect models with sim-to-real epistemic gaps."""
    def __init__(self, belief_state: BeliefState):
        self.belief = belief_state

    def predict_properties(self, params: ParameterPacket, goal: GoalSpecification) -> Dict[str, float]:
        stiffness_coefficient = 12.5 * (1.0 + self.belief.surface_friction_coefficient * 0.1)
        predicted_force = stiffness_coefficient * (params.max_velocity_mps ** 1.5)

        effective_viscosity = self.belief.fluid_viscosity_pas * (1.0 - (self.belief.ambient_temperature_c - 20.0) * 0.01)
        radius = 0.05
        wave_height_m = (params.tilt_rate_radps * radius) / effective_viscosity
        predicted_wave = wave_height_m * 1000.0

        # Injecting minor epistemic simulation error (Sim-to-Real gap)
        epistemic_force_error = random.uniform(-0.2, 0.3)
        epistemic_wave_error = random.uniform(-0.1, 0.2)

        return {
            "predicted_force_n": round(max(0.0, predicted_force + epistemic_force_error), 2),
            "predicted_wave_mm": round(max(0.0, predicted_wave + epistemic_wave_error), 2),
            "predicted_nav_velocity": min(params.max_velocity_mps, goal.max_nav_velocity_mps)
        }


class WarrantGate:
    """Validates predicted properties against the interpreted GoalSpecification."""
    def __init__(self, sdt: SemanticDigitalTwin):
        self.sdt = sdt

    def validate(self, params: ParameterPacket, goal: GoalSpecification, belief: BeliefState) -> Dict[str, Any]:
        predictions = self.sdt.predict_properties(params, goal)

        checks = {
            "grounded": belief.vial_ready and belief.pouch_present,
            "feasible": params.max_velocity_mps > 0.0 and params.tilt_rate_radps > 0.0,
            "safe": (predictions["predicted_force_n"] <= goal.max_allowable_force_n) and
                    (predictions["predicted_wave_mm"] <= goal.max_slosh_wave_mm) and
                    (predictions["predicted_nav_velocity"] <= goal.max_nav_velocity_mps),
            "monitorable": belief.camera_active,
            "repairable": belief.fallback_path_exists,
            "recordable": belief.audit_logger_enabled
        }

        all_passed = all(checks.values())
        failed_criterion = next((name for name, passed in checks.items() if not passed), None)

        return {
            "passed": all_passed,
            "failed_criterion": failed_criterion,
            "predictions": predictions,
            "warrant_checks": checks
        }


class ExecutionEngine:
    """Manages physical execution and constructs the epistemic audit trail."""
    def __init__(self):
        self.audit_trail: List[Dict[str, Any]] = []

    def execute_action(self, params: ParameterPacket, goal: GoalSpecification, warrant_result: Dict[str, Any]) -> Dict[str, Any]:
        print(f"\n[ACTUATOR] Executing motion in physical world: v_max={params.max_velocity_mps} m/s")
        time.sleep(0.1)

        # Real-world divergence check
        predictions = warrant_result["predictions"]
        actual_force = predictions["predicted_force_n"] + random.uniform(-0.3, 0.6)
        actual_wave = predictions["predicted_wave_mm"] + random.uniform(-0.2, 0.5)

        spill_occurred = (actual_force > goal.max_allowable_force_n) or (actual_wave > goal.max_slosh_wave_mm)
        status = "FAILURE" if spill_occurred else "SUCCESS"

        execution_outcome = {
            "actual_force_n": round(actual_force, 2),
            "actual_wave_mm": round(actual_wave, 2),
            "actual_volume_transferred_ml": goal.target_volume_ml if not spill_occurred else goal.target_volume_ml - 3.0,
            "spill_occurred": spill_occurred,
            "status": status
        }
        return execution_outcome

    def record_trace(self, asked_utterance: str, goal: GoalSpecification, belief: BeliefState,
                     proposal: ParameterPacket, warrant_result: Dict[str, Any],
                     observation: Observation, committed: bool, execution_outcome: Optional[Dict[str, Any]] = None) -> None:
        record = {
            "timestamp": time.time(),
            "asked": {
                "raw_utterance": asked_utterance,
                "interpreter_output": {
                    "target": goal.target_object,
                    "max_force_limit_n": goal.max_allowable_force_n,
                    "max_nav_velocity": goal.max_nav_velocity_mps
                }
            },
            "belief": {
                "eva_position": belief.eva_position,
                "room_busyness": belief.room_busyness,
                "ambient_temp_c": belief.ambient_temperature_c,
                "surface_friction": belief.surface_friction_coefficient
            },
            "observed": {
                "timestamp": observation.timestamp,
                "measured_temp_c": observation.measured_temperature_c,
                "measured_friction": observation.measured_friction,
                "eva_position": observation.eva_position,
                "room_busyness": observation.room_busyness
            },
            "committed": {
                "action_approved": committed,
                "proposed_parameters": {
                    "max_velocity_mps": proposal.max_velocity_mps,
                    "tilt_rate_radps": proposal.tilt_rate_radps
                },
                "warrant_verdict": warrant_result
            },
            "execution_outcome": execution_outcome
        }
        self.audit_trail.append(record)


# --- Execution Demonstration ---
if __name__ == "__main__":
    knowledge_base = {
        "cups": {
            "blue_ceramic_mug": {"max_force": 4.5, "max_wave": 1.5},
            "plastic_cup": {"max_force": 7.0, "max_wave": 3.0}
        },
        "room_busyness_limits": {
            "low": 1.0,
            "medium": 0.6,
            "high": 0.3
        }
    }

    # 1. Initialize systems
    belief_state = BeliefState(room_busyness="medium", preferred_cup="blue_ceramic_mug")
    sensing = SensingSystem()
    perception = PerceptionSystem(belief_state)
    interpreter = InterpretationSystem(knowledge_base)
    sdt = SemanticDigitalTwin(belief_state)
    warrant_gate = WarrantGate(sdt)
    executive = ExecutionEngine()

    print("=== STEP 1: Sensing & Perception Update ===")
    raw_observation = sensing.sense(
        true_ambient_temp=21.5,
        true_friction=0.45,
        eva_pos="living_room",
        busyness="medium"
    )
    perception.update(raw_observation)
    print(f"Raw Sensor Observation : {raw_observation}")
    print(f"Updated Internal Belief : Temp={belief_state.ambient_temperature_c}°C, Friction={belief_state.surface_friction_coefficient}")

    print("\n=== STEP 2: Instruction Interpretation ===")
    raw_command = "Bring me my coffee fast!"
    goal = interpreter.interpret(raw_command, belief_state)

    print("\n=== STEP 3: System 1 Proposer Generates Parameters ===")
    proposal_v1 = ParameterPacket(
        grasp_pose=(0.45, 0.12, 0.30),
        max_velocity_mps=0.8,
        tilt_rate_radps=0.2
    )
    print(f"Proposer suggests: v_max = {proposal_v1.max_velocity_mps} m/s")

    print("\n=== STEP 4: System 2 SDT & Warrant Gate Validation (Attempt 1) ===")
    warrant_v1 = warrant_gate.validate(proposal_v1, goal, belief_state)
    print(f"SDT Predictions: Nav-Vel = {warrant_v1['predictions']['predicted_nav_velocity']} m/s")
    print(f"Warrant Checks  : {warrant_v1['warrant_checks']}")
    print(f"Gate Verdict    : REJECTED (Failed on: {warrant_v1['failed_criterion']})")

    executive.record_trace(raw_command, goal, belief_state, proposal_v1, warrant_v1, raw_observation, committed=False)

    print("\n=== STEP 5: Proposer Resamples (Attempt 2) ===")
    proposal_v2 = ParameterPacket(
        grasp_pose=(0.45, 0.12, 0.30),
        max_velocity_mps=0.45,
        tilt_rate_radps=0.08
    )

    warrant_v2 = warrant_gate.validate(proposal_v2, goal, belief_state)
    print(f"SDT Predictions: Force = {warrant_v2['predictions']['predicted_force_n']} N, Wave = {warrant_v2['predictions']['predicted_wave_mm']} mm")
    print(f"Warrant Checks  : {warrant_v2['warrant_checks']}")
    print(f"Gate Verdict    : APPROVED")

    print("\n=== STEP 6: Physical Execution & Epistemic Audit Trail ===")
    if warrant_v2["passed"]:
        final_outcome = executive.execute_action(proposal_v2, goal, warrant_v2)
        executive.record_trace(raw_command, goal, belief_state, proposal_v2, warrant_v2, raw_observation, committed=True, execution_outcome=final_outcome)
        print(f"Execution Outcome: {final_outcome}")

    print(f"\n--- COMPLETE EPISODIC AUDIT TRAIL ({len(executive.audit_trail)} Entries) ---")
    for idx, trace in enumerate(executive.audit_trail, 1):
        status = trace['committed']['action_approved']
        outcome = trace.get('execution_outcome')
        outcome_str = f" | Outcome: {outcome['status']} (Spill: {outcome['spill_occurred']})" if outcome else ""
        print(f"Entry #{idx} | Approved: {status}{outcome_str} | Command: '{trace['asked']['raw_utterance']}'")

=== STEP 1: Sensing & Perception Update ===
Raw Sensor Observation : Observation(timestamp=1793656356.7949362, measured_temperature_c=21.63, measured_friction=0.46, eva_position='living_room', room_busyness='medium')
Updated Internal Belief : Temp=21.63°C, Friction=0.46

=== STEP 2: Instruction Interpretation ===

[INTERPRETATION] Analyzing instruction: 'Bring me my coffee fast!' in context of Eva (Position: living_room, Room Busyness: medium)
 -> Derived Goal: Object=blue_ceramic_mug, MaxForce=4.5N, MaxNavVel=0.72m/s, TimeFrame=30.0s

=== STEP 3: System 1 Proposer Generates Parameters ===
Proposer suggests: v_max = 0.8 m/s

=== STEP 4: System 2 SDT & Warrant Gate Validation (Attempt 1) ===
SDT Predictions: Nav-Vel = 0.72 m/s
Warrant Checks  : {'grounded': True, 'feasible': True, 'safe': False, 'monitorable': True, 'repairable': True, 'recordable': True}
Gate Verdict    : REJECTED (Failed on: safe)

=== STEP 5: Proposer Resamples (Attempt 2) ===
SDT Predictions: Force = 4.11 N, Wave = 

## Questions (0.25 Pt x 4 + 0.5 Pt)

### Task 1: Architectural Mapping & Code Decomposition
* **Question 1.1:** Map the classes and methods implemented in your Python script directly onto the functional blocks of the lecture integration diagram. Explain how data flows between these components during a single cognitive decision cycle.
* **Question 1.2:** Complete the proposer class based on a probabilistic model of your choice (e.g., uniform, gaussian, mixture). Justify your choice.
  

### Task 2: Non-Linear Contact Mechanics and Material Compliance
In the base script, the contact force on the flexible pouch is modeled using a simplified power-law equation.
* **Question 2.1:** Modify the `SemanticDigitalTwin.predict_properties` method to account for a dual-stage material compliance curve. If the contact depth exceeds $0.02 \, \text{m}$, the stiffness coefficient should increase exponentially by a factor of $1.8$ due to bottoming out against a rigid internal fixture.
* **Question 2.2:** Write a test case where the Proposer generates an aggressive approach velocity ($0.35 \, \text{m/s}$). Calculate the predicted force and verify that the Warrant Gate successfully intercepts and rejects the parameter packet before physical execution.

### Task 3: Advanced Fluid Sloshing and Viscosity Temperature Coupling
Fluids in laboratory automation often exhibit temperature-dependent viscosity changes.
* **Question 3.1:** Enhance the fluid dynamics model so that ambient temperature fluctuations directly alter the effective fluid viscosity using an Arrhenius-type approximation or a linear thermal degradation coefficient ($1\% \, \text{drop per} \, ^\circ\text{C}$ above $20^\circ\text{C}$).
* **Question 3.2:** Simulate an environment where the ambient temperature rises to $28^\circ\text{C}$. How does this change the predicted wave height for a tilt rate of $0.15 \, \text{rad/s}$? Does the Warrant Gate trigger a safety violation?

### Task 4: Expanding the Warrant Gate Checklist
The current Warrant Gate evaluates six core criteria (*grounded, feasible, safe, monitorable, repairable, recordable*).
* **Question 4.1:** Add a seventh criterion called **`energetically_viable`** to the Warrant Gate dictionary checks. It must evaluate whether the estimated battery drain for the requested trajectory stays below $15\%$ of total capacity.
* **Question 4.2:** Introduce a low-battery state into the `BeliefState` and demonstrate how a failed `energetically_viable` check redirects the system workflow to request a recharge action rather than proceeding with the coffee delivery.

### Task 5: Post-Hoc Epistemic Audit Analysis
A core pillar of trustworthy robotics is the ability to audit *why* a decision was made or rejected.
* **Question 5.1:** Write a post-processing function (`audit_analyzer(audit_trail)`) that parses the recorded episodic log. It should compute and print:
  1. The total number of rejected proposals versus approved executions.
  2. The most frequent warrant criterion responsible for rejection.
  3. The average variance between predicted forces and observed physical outcomes.
* **Question 5.2:** Discuss in your lab report: *Why does maintaining separate records for `asked`, `belief`, `committed`, and `observed` solve the epistemological gap that pure data-driven imitation learning cannot address?*

Question 1.1: Architectural Mapping & Data Flow
The classes and methods in the Python script directly implement the functional blocks of the Manipulation Executive:   InterpretationSystem.interpret $\rightarrow$ Interpret & Ground: Translates the raw natural language string ("Bring me my coffee fast!") into a precise GoalSpecification, mapping vague requests to concrete object targets and physical limits based on the context of the internal belief state.   Proposer $\rightarrow$ Propose (System 1 - Learning): Generates learned candidate actions (the ParameterPacket) representing the "best guess" for execution parameters.   SensingSystem.sense & PerceptionSystem.update $\rightarrow$ Perceive: Translates raw, noisy sensor readings into an updated BeliefState representing the robot's understanding of the current scene.   SemanticDigitalTwin.predict_properties $\rightarrow$ Know & Predict (System 2): Acts as the internal physics simulator, taking the proposed parameters and calculating the expected physical consequences (force, slosh wave) before they occur in the real world.   WarrantGate.validate $\rightarrow$ Decide (The Warrant Gate): Checks the predicted consequences against the derived goal's safety constraints to determine if the proposal is licensed to move the physical body.   ExecutionEngine.execute_action $\rightarrow$ Act & Regulate: Interacts with the physical Body-World System to execute the approved motion.   ExecutionEngine.record_trace $\rightarrow$ Record & Justify: Compiles the episodic log of what was asked, believed, committed, and observed.   During a single cognitive decision cycle, data flows linearly but with loops: SensingSystem emits an Observation to update the BeliefState. The InterpretationSystem uses this belief to output a GoalSpecification. The Proposer samples a ParameterPacket. The WarrantGate routes this packet through the SemanticDigitalTwin for physics predictions; if the predictions violate the goal limits, the gate rejects the packet and forces the Proposer to resample. Once a packet passes, it is sent to the ExecutionEngine, which returns the physical outcome to be logged.Question 1.2: Completing the Proposer ClassA Gaussian (Normal) distribution is the most mathematically justified probabilistic model for this component. Robotic manipulation execution parameters (such as optimal grasp coordinates or approach velocities) typically cluster tightly around an optimal learned mean, with naturally occurring symmetric variance representing real-world stochastic sampling or imitation learning noise.

Task 2: Non-Linear Contact Mechanics and Material Compliance
Question 2.1: Modifying the Semantic Digital Twin
Question 2.2: Aggressive Approach Velocity Test CaseGiven a proposed maximum velocity of $0.35$ m/s, the simulated contact depth exceeds $0.02$ m.The base stiffness coefficient is calculated as $12.5 \times (1.0 + 0.45 \times 0.1) = 13.0625$.Because the contact depth exceeds the $0.02$ m threshold, the bottoming-out multiplier is applied: $13.0625 \times 1.8 = 23.5125$.The predicted force is calculated using the power-law equation: $23.5125 \times (0.35)^{1.5} \approx 23.5125 \times 0.20706 \approx 4.868$ N.The semantic knowledge base defines the max_force limit for the blue_ceramic_mug as $4.5$ N.Because $4.868 \text{ N} > 4.5 \text{ N}$, the predicted force exceeds the maximum allowable force. The Warrant Gate will successfully intercept and reject this parameter packet before physical execution.

Task 3: Advanced Fluid Sloshing and Viscosity Temperature Coupling

Question 3.1: Enhancing the Fluid Dynamics Model

Question 3.2

Viscosity Change: At $28^\circ\text{C}$, the temperature exceeds the threshold by $8^\circ\text{C}$. Using the 1% thermal degradation coefficient, the viscosity drops by 8% (a factor of $0.92$). The effective fluid viscosity decreases from $1.2 \text{ Pa·s}$ to $1.104 \text{ Pa·s}$.   Wave Height Impact: Because the fluid is less viscous (thinner), it sloshes more easily. For a tilt rate of $0.15 \text{ rad/s}$, the predicted wave height jumps to approximately $6.79 \text{ mm}$ (plus or minus minor epistemic simulation noise).   Warrant Gate Verdict: The predicted wave of $\sim 6.79 \text{ mm}$ significantly exceeds the $1.5 \text{ mm}$ maximum slosh limit defined for the blue_ceramic_mug. The Warrant Gate successfully detects this physical risk, evaluates the safe criterion as False, and triggers a safety violation to reject the parameter packet before execution.

Task 4: Expanding the Warrant Gate Checklist

Task 4.1

Task 5 audit_analyzer

Task 5.1: Post-Hoc Epistemic Audit Analysis

Question 5.2: Epistemological Gap in Pure Imitation Learning
Maintaining separate records for what was asked, the internal belief, the committed parameters, and the observed outcome solves the epistemological gap by providing an auditable causal trail. Pure data-driven imitation learning models map directly from observations to actions without building an internal picture of the situation or predicting consequences. This creates a critical asymmetry: a wrong physical motion takes effect and causes damage before it can be judged or corrected.   By recording the explicit belief state and the digital twin's predictions independently of the raw observations, engineers can audit exactly why a robot committed to an action. If the robot spills the coffee, the separated audit trail reveals whether the failure was caused by a faulty sensor reading, a flawed digital twin simulation (sim-to-real gap), or an interpretation error. This converts an opaque failure into a scientifically measurable and repairable mechanism, making the system's intelligence explainable.

In [7]:
#!/usr/bin/env python3
"""Embodied Cognitive Architecture (Consolidated & Debugged)"""

from dataclasses import dataclass
from typing import Dict, Any, Tuple, Optional, List
import random
import time
from collections import Counter

@dataclass(frozen=True)
class GoalSpecification:
    target_object: str
    destination: str
    target_volume_ml: float
    max_allowable_force_n: float
    max_slosh_wave_mm: float
    max_nav_velocity_mps: float
    time_frame_seconds: float

@dataclass
class ParameterPacket:
    grasp_pose: Tuple[float, float, float]
    max_velocity_mps: float
    tilt_rate_radps: float
    source_generator: str = "stochastic_sampler"

class Proposer:
    """System 1 (The Proposer): Generates execution parameters using a truncated Normal distribution."""
    def __init__(self, base_velocity_mean: float = 0.5, base_tilt_mean: float = 0.1):
        self.base_velocity_mean = base_velocity_mean
        self.base_tilt_mean = base_tilt_mean

    def generate_proposal(self) -> ParameterPacket:
        v_max = max(0.05, random.gauss(self.base_velocity_mean, 0.15))
        tilt = max(0.01, random.gauss(self.base_tilt_mean, 0.05))
        return ParameterPacket(
            grasp_pose=(0.45, 0.12, 0.30),
            max_velocity_mps=round(v_max, 3),
            tilt_rate_radps=round(tilt, 3)
        )

@dataclass
class Observation:
    timestamp: float
    measured_temperature_c: float
    measured_friction: float
    eva_position: str
    room_busyness: str

@dataclass
class BeliefState:
    pouch_present: bool = True
    pouch_material: str = "flexible_plastic"
    fluid_viscosity_pas: float = 1.2
    vial_ready: bool = True
    camera_active: bool = True
    fallback_path_exists: bool = True
    audit_logger_enabled: bool = True
    ambient_temperature_c: float = 21.5
    surface_friction_coefficient: float = 0.45
    eva_position: str = "living_room"
    room_busyness: str = "medium"
    preferred_cup: str = "blue_ceramic_mug"
    battery_level_pct: float = 80.0  # Fixed: Default to healthy battery

class InterpretationSystem:
    def __init__(self, knowledge_base: Dict[str, Any]):
        self.kb = knowledge_base

    def interpret(self, raw_utterance: str, belief: BeliefState) -> GoalSpecification:
        cup_type = belief.preferred_cup
        cup_properties = self.kb["cups"].get(cup_type, {"max_force": 5.0, "max_wave": 2.0})
        busyness_factor = self.kb["room_busyness_limits"].get(belief.room_busyness, 0.5)

        return GoalSpecification(
            target_object=cup_type,
            destination=belief.eva_position,
            target_volume_ml=12.0 if "fast" in raw_utterance.lower() else 15.0,
            max_allowable_force_n=cup_properties["max_force"],
            max_slosh_wave_mm=cup_properties["max_wave"],
            max_nav_velocity_mps=1.2 * busyness_factor,
            time_frame_seconds=30.0 if "fast" in raw_utterance.lower() else 60.0
        )

class SensingSystem:
    def sense(self, true_ambient_temp: float, true_friction: float, eva_pos: str, busyness: str) -> Observation:
        return Observation(
            timestamp=time.time(),
            measured_temperature_c=round(true_ambient_temp + random.uniform(-0.5, 0.5), 2),
            measured_friction=round(true_friction + random.uniform(-0.05, 0.05), 2),
            eva_position=eva_pos,
            room_busyness=busyness
        )

class PerceptionSystem:
    def __init__(self, belief_state: BeliefState):
        self.belief_state = belief_state

    def update(self, observation: Observation) -> None:
        self.belief_state.ambient_temperature_c = observation.measured_temperature_c
        self.belief_state.surface_friction_coefficient = observation.measured_friction
        self.belief_state.eva_position = observation.eva_position
        self.belief_state.room_busyness = observation.room_busyness

class SemanticDigitalTwin:
    def __init__(self, belief_state: BeliefState):
        self.belief = belief_state

    def predict_properties(self, params: ParameterPacket, goal: GoalSpecification) -> Dict[str, float]:
        # Task 2.1: Dynamic contact depth simulation proxy based on velocity
        simulated_contact_depth_m = params.max_velocity_mps * 0.1
        stiffness_coefficient = 12.5 * (1.0 + self.belief.surface_friction_coefficient * 0.1)
        if simulated_contact_depth_m > 0.02:
            stiffness_coefficient *= 1.8

        predicted_force = stiffness_coefficient * (params.max_velocity_mps ** 1.5)

        # Task 3.1: Viscosity Temperature Coupling (Base 20°C)
        temp_difference = max(0.0, self.belief.ambient_temperature_c - 20.0)
        thermal_degradation_factor = max(0.1, 1.0 - (temp_difference * 0.01))
        effective_viscosity = self.belief.fluid_viscosity_pas * thermal_degradation_factor

        wave_height_m = (params.tilt_rate_radps * 0.05) / effective_viscosity

        return {
            "predicted_force_n": round(max(0.0, predicted_force + random.uniform(-0.1, 0.1)), 2),
            "predicted_wave_mm": round(max(0.0, (wave_height_m * 1000.0) + random.uniform(-0.1, 0.1)), 2),
            "predicted_nav_velocity": min(params.max_velocity_mps, goal.max_nav_velocity_mps)
        }

class WarrantGate:
    def __init__(self, sdt: SemanticDigitalTwin):
        self.sdt = sdt

    def validate(self, params: ParameterPacket, goal: GoalSpecification, belief: BeliefState) -> Dict[str, Any]:
        predictions = self.sdt.predict_properties(params, goal)
        # Task 4.1: Assumption - Drain stays below 15% total capacity AND battery has enough charge
        estimated_drain_pct = (params.max_velocity_mps * 10.0) + 2.0

        checks = {
            "grounded": belief.vial_ready and belief.pouch_present,
            "feasible": params.max_velocity_mps > 0.0 and params.tilt_rate_radps > 0.0,
            "safe": (predictions["predicted_force_n"] <= goal.max_allowable_force_n) and
                    (predictions["predicted_wave_mm"] <= goal.max_slosh_wave_mm),
            "monitorable": belief.camera_active,
            "repairable": belief.fallback_path_exists,
            "recordable": belief.audit_logger_enabled,
            "energetically_viable": (belief.battery_level_pct - estimated_drain_pct) > 15.0
        }

        # Fix B2: Return ALL failed criteria
        failed_criteria = [name for name, passed in checks.items() if not passed]

        return {
            "passed": all(checks.values()),
            "failed_criteria": failed_criteria,
            "predictions": predictions,
            "warrant_checks": checks
        }

class ExecutionEngine:
    def __init__(self):
        self.audit_trail: List[Dict[str, Any]] = []

    def execute_action(self, params: ParameterPacket, goal: GoalSpecification, warrant_result: Dict[str, Any]) -> Dict[str, Any]:
        predictions = warrant_result["predictions"]
        actual_force = predictions["predicted_force_n"] + random.uniform(-0.2, 0.3)
        spill_occurred = (actual_force > goal.max_allowable_force_n)

        return {
            "actual_force_n": round(actual_force, 2),
            "status": "FAILURE" if spill_occurred else "SUCCESS"
        }

    def record_trace(self, utterance: str, goal: GoalSpecification, belief: BeliefState, proposal: ParameterPacket, warrant: Dict[str, Any], obs: Observation, committed: bool, outcome: Optional[Dict[str, Any]] = None) -> None:
        self.audit_trail.append({
            "asked": utterance, "committed": {"action_approved": committed, "warrant_verdict": warrant},
            "execution_outcome": outcome
        })

def audit_analyzer(audit_trail: List[Dict[str, Any]]) -> None:
    if not audit_trail:
        return
    rejected = sum(1 for t in audit_trail if not t["committed"]["action_approved"])
    approved = sum(1 for t in audit_trail if t["committed"]["action_approved"])

    reasons = []
    for t in audit_trail:
        if not t["committed"]["action_approved"]:
            reasons.extend(t["committed"]["warrant_verdict"]["failed_criteria"])

    freq_rejection = Counter(reasons).most_common(1)[0][0] if reasons else "None"

    # Task 5.1: Compute actual Variance, not MAE
    errors = [t["committed"]["warrant_verdict"]["predictions"]["predicted_force_n"] - t["execution_outcome"]["actual_force_n"]
              for t in audit_trail if t["committed"]["action_approved"] and t.get("execution_outcome")]

    variance = (sum((x - (sum(errors)/len(errors)))**2 for x in errors) / len(errors)) if errors else 0.0

    print(f"\n--- AUDIT ANALYZER ---")
    print(f"Rejected: {rejected} | Approved: {approved} | Most Frequent Rejection: {freq_rejection}")
    print(f"Force Sim-to-Real Variance: {variance:.3f} N^2")

# ==========================================
# MASTER EXECUTION SCRIPT
# ==========================================
if __name__ == "__main__":
    kb = {"cups": {"blue_ceramic_mug": {"max_force": 4.5, "max_wave": 1.5}}, "room_busyness_limits": {"medium": 0.6}}
    belief = BeliefState()
    interpreter = InterpretationSystem(kb)
    sdt = SemanticDigitalTwin(belief)
    gate = WarrantGate(sdt)
    executive = ExecutionEngine()

    goal = interpreter.interpret("Bring me my coffee fast!", belief)
    obs = SensingSystem().sense(21.5, 0.45, "living_room", "medium")
    PerceptionSystem(belief).update(obs)

    # ------------------------------------------
    # DEMO 1: Standard Execution (Fixing B1)
    # ------------------------------------------
    print("\n=== DEMO 1: Base Execution ===")
    prop_fail = ParameterPacket((0.45, 0.12, 0.3), 0.8, 0.2)
    war_fail = gate.validate(prop_fail, goal, belief)
    print(f"Attempt 1 (v=0.8) Verdict: {'APPROVED' if war_fail['passed'] else 'REJECTED on ' + str(war_fail['failed_criteria'])}")
    executive.record_trace("Bring coffee", goal, belief, prop_fail, war_fail, obs, False)

    prop_pass = ParameterPacket((0.45, 0.12, 0.3), 0.25, 0.03) # Safe params
    war_pass = gate.validate(prop_pass, goal, belief)
    print(f"Attempt 2 (v=0.25) Verdict: {'APPROVED' if war_pass['passed'] else 'REJECTED'}")
    if war_pass["passed"]:
        out = executive.execute_action(prop_pass, goal, war_pass)
        executive.record_trace("Bring coffee", goal, belief, prop_pass, war_pass, obs, True, out)

    # ------------------------------------------
    # DEMO 2: Task 2.2 Aggressive Approach
    # ------------------------------------------
    print("\n=== DEMO 2: Task 2.2 Aggressive Contact ===")
    prop_aggro = ParameterPacket((0.45, 0.12, 0.3), 0.35, 0.02) # High V, Safe tilt
    war_aggro = gate.validate(prop_aggro, goal, belief)
    print(f"Predicted Force: {war_aggro['predictions']['predicted_force_n']} N | Limit: {goal.max_allowable_force_n} N")
    print(f"Rejected By: {war_aggro['failed_criteria']}")
    executive.record_trace("Bring coffee", goal, belief, prop_aggro, war_aggro, obs, False)

    # ------------------------------------------
    # DEMO 3: Task 3.2 Viscosity/Temperature
    # ------------------------------------------
    print("\n=== DEMO 3: Task 3.2 Heat Slosh (20C vs 28C) ===")
    prop_heat = ParameterPacket((0.45, 0.12, 0.3), 0.1, 0.15)
    belief.ambient_temperature_c = 20.0
    wave_20 = gate.validate(prop_heat, goal, belief)["predictions"]["predicted_wave_mm"]

    belief.ambient_temperature_c = 28.0
    war_heat = gate.validate(prop_heat, goal, belief)
    wave_28 = war_heat["predictions"]["predicted_wave_mm"]
    print(f"Wave at 20C: {wave_20} mm -> Wave at 28C: {wave_28} mm (Delta: {round(wave_28 - wave_20, 2)} mm)")
    print(f"Verdict at 28C: {war_heat['failed_criteria']}")
    executive.record_trace("Bring coffee", goal, belief, prop_heat, war_heat, obs, False)

    # ------------------------------------------
    # DEMO 4: Task 4.2 Low Battery Redirect (Fixing B2 & B3)
    # ------------------------------------------
    print("\n=== DEMO 4: Task 4.2 Low Battery ===")
    belief.ambient_temperature_c = 21.5 # Reset temperature from Demo 3!
    belief.battery_level_pct = 12.0     # Force critical battery state

    # Ultra-slow parameters (v=0.1, tilt=0.01) so 'safe' passes easily
    prop_safe_motion = ParameterPacket((0.45, 0.12, 0.3), 0.1, 0.01)
    war_batt = gate.validate(prop_safe_motion, goal, belief)

    print(f"Failed On: {war_batt['failed_criteria']}")

    if not war_batt["passed"] and "energetically_viable" in war_batt["failed_criteria"]:
        print("[WORKFLOW REDIRECT] Battery critical. Aborting coffee delivery. Triggering RechargeAction()...")
    executive.record_trace("Bring coffee", goal, belief, prop_safe_motion, war_batt, obs, False)

    # Output Final Audit
    audit_analyzer(executive.audit_trail)


=== DEMO 1: Base Execution ===
Attempt 1 (v=0.8) Verdict: REJECTED on ['safe']
Attempt 2 (v=0.25) Verdict: APPROVED

=== DEMO 2: Task 2.2 Aggressive Contact ===
Predicted Force: 4.9 N | Limit: 4.5 N
Rejected By: ['safe']

=== DEMO 3: Task 3.2 Heat Slosh (20C vs 28C) ===
Wave at 20C: 6.25 mm -> Wave at 28C: 6.86 mm (Delta: 0.61 mm)
Verdict at 28C: ['safe']

=== DEMO 4: Task 4.2 Low Battery ===
Failed On: ['energetically_viable']
[WORKFLOW REDIRECT] Battery critical. Aborting coffee delivery. Triggering RechargeAction()...

--- AUDIT ANALYZER ---
Rejected: 4 | Approved: 1 | Most Frequent Rejection: safe
Force Sim-to-Real Variance: 0.000 N^2


## Part 3: Robot Anatomy (0.5 Pt)

**Title:** Links Vs Joints, Actuators Vs Sensors, Effectors Vs Bases Vs Torsos Vs Heads, Wheels Vs Legs Vs Grippers, Proprioception Vs Exteroception Vs Interoception

![Alternative Text](robot-body.jpg)

### Questions

Identify the key functional parts of a robot based on the above picture and briefly indicate how they support the execution of function in the above architecture.

---

**Part 3: Robot Anatomy**
Based on the provided `robot-body.jpg` diagram (showing a PR2 mobile manipulator):
*   **Links and Joints / Actuators and Sensors:** The blue callout lines highlight the robot's rigid arm links and an internal electric motor (actuator). These actuators physically drive the joints to execute the movement commands (like `max_velocity_mps`) issued by the `ExecutionEngine`. Sensors (like joint encoders) capture physical measurements for the `SensingSystem`.
*   **Bases (Wheels):** The bottom-left blue callout explicitly shows an omnidirectional caster wheel inside the base, which handles locomotion to navigate the home environment and reach the goal destination.
*   **Torsos & Heads:** The torso provides vertical reach. The red callout lines highlight the head-mounted camera unit (vision), which monitors the environment and directly enables the `monitorable` check within the Warrant Gate.
*   **Effectors (Grippers):** The middle-left blue callout details a parallel-jaw gripper. This acts as the effector, physically grasping the tray and mug while adhering to the simulated `max_allowable_force_n` constraints.
*   **Proprioception, Exteroception, Interoception:**
    *   *Exteroception:* The red lines explicitly highlight two external sensors: the head-mounted camera and a base-mounted LiDAR/laser scanner. These capture the outside world, flowing through the `PerceptionSystem` to update the `BeliefState`.
    *   *Proprioception:* Internal joint encoders in the arm/base ensure the `ExecutionEngine` tracks physical progress against the trajectory.
    *   *Interoception:* Internal battery sensors support viability checks, such as the `energetically_viable` constraint I implemented for the Warrant Gate.

# Submission guidelines

- Deadline: 03.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (Create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put all the answers in the notebook. Any other media related to your answer should be inserted in the notebook as well. The notebook and all related media should then be put into a folder called **Assignment1**.